# 🎨 Zero-Trust Ephemeral Ciphertext Mailbox Worker (Diffusion & Video)

Execute batch image/video inference (`SDXL`, `FLUX.1-schnell`, `Wan 2.1/2.2`) on remote cloud GPUs without writing plaintext prompts, reference images, or outputs to shared cloud disks:
1. Run **Cell 1** once to install dependencies and enter your passphrase into memory.
2. Paste `KISSSHOT_TASK_HEX` generated by your local desktop console into **Cell 2**.
3. Pull and decrypt the `WZ_AES` 256-bit `ZIP_STORED` archive locally (burn-after-reading automatically deletes the remote ciphertext blob).


In [ ]:
# === Cell 1: Dependencies & Passphrase Initialization ===
import subprocess, sys, os, io, gc, json, time, secrets, getpass

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "diffusers>=0.33.0", "transformers", "accelerate", "sentencepiece",
    "imageio[ffmpeg]", "cryptography", "pyzipper", "huggingface_hub"
])

import torch, pyzipper
from PIL import Image
from huggingface_hub import HfApi, hf_hub_download
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

KDF_ITERATIONS = 210_000
SALT_BYTES, NONCE_BYTES = 16, 12
_PIPE_CACHE = {"repo": None, "pipe": None}

def derive_key(pwd: str, salt: bytes) -> bytes:
    kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=KDF_ITERATIONS)
    return kdf.derive(pwd.encode("utf-8"))

def decrypt_payload_hex(hex_str: str, pwd: str, aad: bytes = b"kissshot-payload-v1") -> dict:
    raw = bytes.fromhex("".join(hex_str.split()))
    pt = AESGCM(derive_key(pwd, raw[:SALT_BYTES])).decrypt(raw[SALT_BYTES:SALT_BYTES + NONCE_BYTES], raw[SALT_BYTES + NONCE_BYTES:], aad)
    return json.loads(pt.decode("utf-8"))

def decrypt_bytes_aes(enc: bytes, pwd: str, aad: bytes = b"kissshot-blob-v1") -> bytes:
    return AESGCM(derive_key(pwd, enc[:SALT_BYTES])).decrypt(enc[SALT_BYTES:SALT_BYTES + NONCE_BYTES], enc[SALT_BYTES + NONCE_BYTES:], aad)

def pack_encrypted_zip(items: list, pwd: str, meta: dict) -> bytes:
    buf = io.BytesIO()
    pwd_b = pwd.encode("utf-8")
    manifest = {"version": 1, "created_at": int(time.time()), "files": [], "meta": meta}
    with pyzipper.AESZipFile(buf, "w", compression=pyzipper.ZIP_STORED, encryption=pyzipper.WZ_AES) as zf:
        zf.setpassword(pwd_b)
        zf.setencryption(pyzipper.WZ_AES, nbits=256)
        for i, (ext, data) in enumerate(items):
            inner = f"chunk_{i:03d}.dat"
            zf.writestr(inner, data)
            manifest["files"].append({"inner": inner, "ext": ext.lstrip(".")})
        zf.writestr("manifest.json", json.dumps(manifest, ensure_ascii=False).encode("utf-8"))
    return buf.getvalue()

SESSION_PASSPHRASE = getpass.getpass("🔑 Enter your local ZTGB passphrase (in-memory only): ").strip()
assert SESSION_PASSPHRASE, "Passphrase cannot be empty."
print("✅ Worker ready on GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# === Cell 2: Decrypt KISSSHOT_TASK_HEX, Run In-Memory Inference & Relay WZ_AES Archive ===
KISSSHOT_TASK_HEX = ""

assert KISSSHOT_TASK_HEX.strip(), "Paste KISSSHOT_TASK_HEX from your local console first."
task = decrypt_payload_hex(KISSSHOT_TASK_HEX, SESSION_PASSPHRASE)
assert task.get("type") == "kissshot_media_infer", "Invalid task payload type."

target_repo = task["src_repo"]
arch = task["arch"]
hf_token = task["hf_token"]
mailbox_repo = task["mailbox_repo"]
api = HfApi(token=hf_token)

if _PIPE_CACHE["repo"] != target_repo:
    if _PIPE_CACHE["pipe"] is not None:
        del _PIPE_CACHE["pipe"]
        _PIPE_CACHE["pipe"] = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if arch == "sdxl":
        from diffusers import StableDiffusionXLPipeline
        kw = {"torch_dtype": torch.float16, "token": hf_token, "use_safetensors": True}
        if task.get("variant"):
            kw["variant"] = task["variant"]
        pipe = StableDiffusionXLPipeline.from_pretrained(target_repo, **kw).to("cuda")
    elif arch == "flux":
        from diffusers import FluxPipeline
        pipe = FluxPipeline.from_pretrained(target_repo, torch_dtype=torch.bfloat16, token=hf_token)
        pipe.enable_model_cpu_offload()
    elif arch in ("wan_t2v", "wan_ti2v"):
        from diffusers import WanPipeline, AutoencoderKLWan
        vae = AutoencoderKLWan.from_pretrained(target_repo, subfolder="vae", torch_dtype=torch.float32, token=hf_token)
        pipe = WanPipeline.from_pretrained(target_repo, vae=vae, torch_dtype=torch.bfloat16, token=hf_token)
        pipe.enable_model_cpu_offload()
    else:
        raise ValueError(f"Unsupported architecture: {arch}")
    _PIPE_CACHE["repo"] = target_repo
    _PIPE_CACHE["pipe"] = pipe

pipe = _PIPE_CACHE["pipe"]
seed = task["seed"] if task["seed"] >= 0 else secrets.randbits(31)
width, height = (int(task["width"]) // 16) * 16, (int(task["height"]) // 16) * 16
steps, cfg_val = int(task["steps"]), float(task["cfg"])
out_blobs = []

if arch in ("sdxl", "flux"):
    for idx in range(max(1, int(task.get("batch", 1)))):
        gen = torch.Generator(device="cpu").manual_seed(seed + idx)
        kw = dict(prompt=task["prompt"], width=width, height=height, num_inference_steps=steps, guidance_scale=cfg_val, generator=gen)
        if arch == "sdxl":
            kw["negative_prompt"] = task["negative_prompt"]
        img = pipe(**kw).images[0]
        mem = io.BytesIO()
        img.save(mem, format="PNG")
        out_blobs.append(("png", mem.getvalue()))
else:
    import tempfile
    from diffusers.utils import export_to_video
    frames_cnt = ((max(17, int(task.get("frames", 81))) - 1) // 4) * 4 + 1
    gen = torch.Generator(device="cpu").manual_seed(seed)
    frames = pipe(prompt=task["prompt"], negative_prompt=task["negative_prompt"], width=width, height=height, num_frames=frames_cnt, guidance_scale=cfg_val, num_inference_steps=steps, generator=gen).frames[0]
    shm = "/dev/shm" if os.path.isdir("/dev/shm") else tempfile.gettempdir()
    tmp_mp4 = os.path.join(shm, f"ztgb_{secrets.token_hex(6)}.mp4")
    try:
        export_to_video(frames, tmp_mp4, fps=16)
        with open(tmp_mp4, "rb") as vf:
            out_blobs.append(("mp4", vf.read()))
    finally:
        if os.path.exists(tmp_mp4):
            os.remove(tmp_mp4)

enc_zip = pack_encrypted_zip(out_blobs, SESSION_PASSPHRASE, meta={"seed": seed, "preset": task["preset_key"]})
remote_blob = f"_inbox/pkt_{int(time.time())}_{secrets.token_hex(4)}.bin"
api.create_repo(repo_id=mailbox_repo, repo_type="model", private=True, exist_ok=True)
api.upload_file(path_or_fileobj=enc_zip, path_in_repo=remote_blob, repo_id=mailbox_repo, repo_type="model", commit_message="relay encrypted artifact")
print(f"✅ Done (seed={seed}). Encrypted archive uploaded to {mailbox_repo}/{remote_blob}. Pull & burn from your local console.")
